In [2]:
import requests
from bs4 import BeautifulSoup
import time
import json
import os
import re

# --- Configuration ---
DATA_FILE = "ewg_exposure_data.json"
PROGRESS_FILE = "category_progress.json"
DOMAIN = "https://www.ewg.org"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

def get_all_categories():
    print("Discovering categories...")
    url = 'https://www.ewg.org/skindeep/'
    try:
        response = requests.get(url, headers=HEADERS, timeout=15)
        soup = BeautifulSoup(response.text, "lxml")
        cat_links = set()
        for link in soup.find_all('a', attrs={'href': re.compile("^/skindeep/browse/")}):
            full_url = DOMAIN + link.get('href')
            cat_links.add(full_url)
        return sorted(list(cat_links))
    except Exception as e:
        print(f"Failed to fetch main directory: {e}")
        return []

def load_existing_data():
    """Loads the main JSON data store."""
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_data(data):
    """Writes the entire category-product mapping to JSON."""
    with open(DATA_FILE, 'w') as f:
        json.dump(data, f, indent=4)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return json.load(f)
    return {"cat_idx": 0, "page": 1}

def save_progress(cat_idx, page):
    with open(PROGRESS_FILE, 'w') as f:
        json.dump({"cat_idx": cat_idx, "page": page}, f)

def scrape_with_provenance():
    categories = get_all_categories()
    progress = load_progress()
    data_store = load_existing_data()
    
    start_cat_idx = progress["cat_idx"]
    start_page = progress["page"]

    for c_idx in range(start_cat_idx, len(categories)):
        cat_url = categories[c_idx]
        
        # Initialize category entry in data_store if not present
        if cat_url not in data_store:
            data_store[cat_url] = []
            
        current_page = start_page if c_idx == start_cat_idx else 1
        print(f"\n--- Category {c_idx + 1}/{len(categories)}: {cat_url} ---")

        while True:
            try:
                response = requests.get(cat_url, params={'page': current_page}, headers=HEADERS, timeout=15)
                
                if response.status_code == 404: # End of pagination
                    break
                if response.status_code != 200:
                    print(f"Stop: Status {response.status_code}")
                    return

                soup = BeautifulSoup(response.text, 'html.parser')
                tiles = soup.find_all('div', class_='product-tile')

                if not tiles:
                    break

                # Extract links and add to the specific category list
                new_links = []
                for t in tiles:
                    a_tag = t.find('a', href=True)
                    if a_tag:
                        full_product_url = DOMAIN + a_tag['href']
                        if full_product_url not in data_store[cat_url]:
                            data_store[cat_url].append(full_product_url)
                            new_links.append(full_product_url)

                print(f"Page {current_page}: Added {len(new_links)} new products to category.")
                
                # Update files immediately
                save_data(data_store)
                current_page += 1
                save_progress(c_idx, current_page)
                
                time.sleep(1.2)

            except Exception as e:
                print(f"Error: {e}")
                return
        
        # Reset page for the next category
        start_page = 1

    print("\nScraping complete. Provenance maintained in JSON.")

if __name__ == "__main__":
    scrape_with_provenance()

Discovering categories...

--- Category 103/140: https://www.ewg.org/skindeep/browse/category/Men/ ---
Page 531: Added 12 new products to category.
Page 532: Added 12 new products to category.
Page 533: Added 12 new products to category.
Page 534: Added 12 new products to category.
Page 535: Added 12 new products to category.
Page 536: Added 12 new products to category.
Page 537: Added 12 new products to category.
Page 538: Added 12 new products to category.
Page 539: Added 12 new products to category.
Page 540: Added 12 new products to category.
Page 541: Added 12 new products to category.
Page 542: Added 12 new products to category.
Page 543: Added 12 new products to category.
Page 544: Added 12 new products to category.
Page 545: Added 12 new products to category.
Page 546: Added 12 new products to category.
Page 547: Added 12 new products to category.
Page 548: Added 12 new products to category.
Page 549: Added 12 new products to category.
Page 550: Added 12 new products to categor

## categories

In [3]:
from __future__ import annotations

import hashlib
import json
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# -----------------------------
# Metadata model
# -----------------------------
@dataclass
class SnapshotMeta:
    category_url: str           # the key in your input dict (category/listing page)
    target_url: str             # URL we attempted to fetch (after normalization)
    fetched_at_utc: str
    final_url: str
    status_code: int
    content_type: str
    encoding: str
    content_sha256: str
    etag: Optional[str]
    last_modified: Optional[str]
    server: Optional[str]
    out_html: str               # relative path to saved html
    out_meta: str               # relative path to saved json meta


# -----------------------------
# Helpers
# -----------------------------
def make_session(
    total_retries: int = 5,
    backoff_factor: float = 0.8,
) -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=total_retries,
        connect=total_retries,
        read=total_retries,
        status=total_retries,
        backoff_factor=backoff_factor,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=("GET", "HEAD"),
        raise_on_status=False,
        respect_retry_after_header=True,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    session.headers.update(
        {
            "User-Agent": "ProvenanceSnapshotBot/1.0 (+contact: you@example.com)",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        }
    )
    return session


def _safe_slug(text: str, max_len: int = 80) -> str:
    text = re.sub(r"[^a-zA-Z0-9._-]+", "_", text).strip("_.-")
    return text[:max_len] if len(text) > max_len else text


def _short_hash(s: str, n: int = 12) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:n]


def normalize_url(u: str) -> str:
    """
    Fix common malformed cases like:
      https://www.ewg.orghttps://www.ewg.org/skindeep/products/...
    Also trims whitespace.

    If a URL is already fine, returns it unchanged.
    """
    u = (u or "").strip()

    # Case: "https://www.ewg.orghttps://www.ewg.org/...."
    # Keep only the *last* occurrence of the scheme+host pattern.
    # This is conservative and works well for the sample.
    m = re.findall(r"https?://[^/]+", u)
    if len(m) >= 2:
        # If it repeats the same host, drop the first part.
        # Example: ["https://www.ewg.org", "https://www.ewg.org"]
        # Split on the second match.
        second = m[1]
        idx = u.find(second)
        if idx != -1:
            u = u[idx:]

    # Also handle accidental "https://www.ewg.orgwww.ewg.org/..."
    u = re.sub(r"^(https?://[^/]+)(https?://)", r"\2", u)

    return u


def iter_targets(input_map: Dict[str, List[str]]) -> Iterable[Tuple[str, str]]:
    """
    Yields (category_url, target_url) pairs.
    """
    for category_url, url_list in input_map.items():
        if not isinstance(url_list, list):
            continue
        for raw in url_list:
            yield category_url, normalize_url(raw)

In [5]:
# -----------------------------
# Main snapshotter
# -----------------------------
def snapshot_from_category_map(
    category_to_urls: Dict[str, List[str]],
    out_dir: str | Path = "provenance_snapshots",
    timeout_seconds: int = 30,
    delay_seconds: float = 0.25,
    overwrite: bool = False,
) -> None:
    out_dir = Path(out_dir)
    html_dir = out_dir / "html"
    meta_dir = out_dir / "meta"
    html_dir.mkdir(parents=True, exist_ok=True)
    meta_dir.mkdir(parents=True, exist_ok=True)

    manifest_path = out_dir / "manifest.jsonl"

    session = make_session()

    targets = list(iter_targets(category_to_urls))
    total = len(targets)

    for n, (category_url, target_url) in enumerate(targets, start=1):
        if not target_url:
            print(f"[{n}/{total}] SKIP empty url in category {category_url}")
            continue

        ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

        parsed = urlparse(target_url)
        domain = _safe_slug(parsed.netloc or "unknown_domain")
        path_slug = _safe_slug(parsed.path.strip("/")) or "root"
        url_id = _short_hash(target_url)

        base_name = f"{ts}__{domain}__{path_slug}__{url_id}"
        html_path = html_dir / f"{base_name}.html"
        meta_path = meta_dir / f"{base_name}.json"

        if not overwrite and html_path.exists() and meta_path.exists():
            print(f"[{n}/{total}] SKIP exists: {html_path.name}")
            continue

        try:
            r = session.get(target_url, timeout=timeout_seconds, allow_redirects=True)

            body_bytes = r.content
            sha = hashlib.sha256(body_bytes).hexdigest()

            encoding = r.encoding or "utf-8"
            try:
                body_text = body_bytes.decode(encoding, errors="replace")
            except LookupError:
                encoding = "utf-8"
                body_text = body_bytes.decode("utf-8", errors="replace")

            # Save HTML (always UTF-8 on disk for uniformity)
            html_path.write_text(body_text, encoding="utf-8", errors="replace")

            meta = SnapshotMeta(
                category_url=category_url,
                target_url=target_url,
                fetched_at_utc=ts,
                final_url=str(r.url),
                status_code=int(r.status_code),
                content_type=r.headers.get("Content-Type", ""),
                encoding=encoding,
                content_sha256=sha,
                etag=r.headers.get("ETag"),
                last_modified=r.headers.get("Last-Modified"),
                server=r.headers.get("Server"),
                out_html=str(html_path.relative_to(out_dir)),
                out_meta=str(meta_path.relative_to(out_dir)),
            )
            meta_path.write_text(json.dumps(asdict(meta), indent=2, sort_keys=True), encoding="utf-8")

            # Append one-line manifest record for easy loading later
            with manifest_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(asdict(meta), sort_keys=True) + "\n")

            print(f"[{n}/{total}] OK {r.status_code} -> {html_path.name}")

        except requests.RequestException as e:
            print(f"[{n}/{total}] ERROR fetching {target_url}: {e}")

        time.sleep(delay_seconds)

In [ ]:
if __name__ == "__main__":
    # Replace this with your full structure (dict of category/listing URLs -> list of pages)
    from pathlib import Path
    import json
    
    # Load category -> product URL mapping from disk
    json_path = Path("ewg_exposure_data.json")
    
    with json_path.open("r", encoding="utf-8") as f:
        CATEGORY_TO_URLS = json.load(f)
    
    # Basic structural validation (fail fast if malformed)
    if not isinstance(CATEGORY_TO_URLS, dict):
        raise ValueError("ewg_exposure_data.json must contain a JSON object (dict)")
    
    for k, v in CATEGORY_TO_URLS.items():
        if not isinstance(k, str) or not isinstance(v, list):
            raise ValueError(
                "Invalid structure in ewg_exposure_data.json. "
                "Expected {category_url: [list_of_product_urls]}"
            )
    
    print(f"Loaded {len(CATEGORY_TO_URLS)} categories "
          f"and {sum(len(v) for v in CATEGORY_TO_URLS.values())} total URLs.")
    
    snapshot_from_category_map(
        CATEGORY_TO_URLS,
        out_dir="ewg_snapshots",
        timeout_seconds=30,
        delay_seconds=0.5,
        overwrite=False,
    )

In [ ]:
from pathlib import Path
import json
from __future__ import annotations

import hashlib
import json
import re
import time
from dataclasses import asdict, dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple
from urllib.parse import urlparse

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# -----------------------------
# Config
# -----------------------------
OUT_DIR = Path("ewg_snapshots")
MANIFEST_PATH = OUT_DIR / "manifest.jsonl"

json_path = Path("ewg_exposure_data.json")
    
with json_path.open("r", encoding="utf-8") as f:
    CATEGORY_TO_URLS = json.load(f)

# -----------------------------
# 1. Load previously completed URLs
# -----------------------------
completed_urls = set()

if MANIFEST_PATH.exists():
    with MANIFEST_PATH.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                record = json.loads(line)
                completed_urls.add(record["target_url"])

print(f"Found {len(completed_urls)} previously downloaded URLs.")

# -----------------------------
# 2. Build filtered restart map
# -----------------------------
restart_map = {}

for category_url, url_list in CATEGORY_TO_URLS.items():
    remaining = []
    
    for raw_url in url_list:
        normalized = normalize_url(raw_url)
        if normalized not in completed_urls:
            remaining.append(raw_url)
    
    if remaining:
        restart_map[category_url] = remaining

total_remaining = sum(len(v) for v in restart_map.values())

print(f"{total_remaining} URLs remaining to download.")

# -----------------------------
# 3. Resume snapshotting
# -----------------------------
if total_remaining == 0:
    print("Nothing left to download.")
else:
    snapshot_from_category_map(
        restart_map,
        out_dir=OUT_DIR,
        timeout_seconds=30,
        delay_seconds=0.5,
        overwrite=False,
    )

Found 41075 previously downloaded URLs.
156112 URLs remaining to download.
[1/156112] OK 500 -> 20260225T141319Z__www.ewg.org__skindeep_products_9077742-Soo_Ae_Sooae_Phyto_Collagen_Essential_Eye_Roller_Serum__9ba07a8d257b.html
[2/156112] OK 200 -> 20260225T141347Z__www.ewg.org__skindeep_products_959205-Grow_Gorgeous_Curl_Defining_Conditioner__5f49a6c78b57.html
[3/156112] OK 200 -> 20260225T141349Z__www.ewg.org__skindeep_products_838229-Verb_Hydrating_Conditioner_2019_formulation__71f573fcbde8.html
[4/156112] OK 200 -> 20260225T141350Z__www.ewg.org__skindeep_products_973826-Signature_Care_Coconut_Milk_Conditioner__abc379c1de5d.html
[5/156112] OK 200 -> 20260225T141351Z__www.ewg.org__skindeep_products_872216-Head__Shoulders_Advanced_2In1_Shampoo__Conditioner_Tea___52a32e7053b8.html
[6/156112] OK 200 -> 20260225T141353Z__www.ewg.org__skindeep_products_1053249-As_I_Am_Dry__Itchy_LeaveIn_Conditioner__ef4bd74bb2ec.html
[7/156112] OK 200 -> 20260225T141355Z__www.ewg.org__skindeep_products_106